In [1]:
import os
import glob
import pandas as pd
import rasterio
import numpy as np

In [2]:
DATA_PATH = "/content/drive/MyDrive/collab/f2/all_merged_csv_without_correlation.csv"
DISTANCE_PATH = "/content/drive/MyDrive/collab/newdataset/distances_grille_portugal.csv"

In [3]:

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

data = pd.read_csv(DATA_PATH)
print(f"Shape         : {data.shape}")
print(f"\nValeurs manquantes par feature :")

data.head(3)

Mounted at /content/drive
Shape         : (89198, 36)

Valeurs manquantes par feature :


,latitude,longitude,landcover,COARSE,SAND,SILT,CLAY,TEXTURE_USDA,TEXTURE_SOTER,BULK,...,tmin_ete,Elevation,dif_perc_hiver_printemps,dif_perc_automne_printemps,dif_perc_ete_automne,dif_max_automne_printemps,dif_max_hiver_printemps,dif_min_hiver_printemps,dif_min_automne_hiver,dif_max_hiver_min_automne
0,36.987823,-7.963676,210,3.0,26.0,33.0,41.0,3.0,F,1.06,...,19.166667,0.0,3.533332,4.049997,-52.333332,1.666667,-3.000000,-2.333333,5.666667,3.333333
1,36.994129,-8.952213,110,5.0,20.0,35.0,45.0,3.0,F,1.55,...,17.500000,0.0,-6.774999,-11.299998,-46.683333,1.750000,-1.416667,-1.416667,4.666667,2.833333
2,36.996850,-7.974890,210,3.0,26.0,33.0,41.0,3.0,F,1.06,...,19.166667,0.0,3.533332,4.049997,-52.333332,1.666667,-3.000000,-2.333333,5.666667,3.333333


In [4]:
distance = pd.read_csv(DISTANCE_PATH)
print(f"Shape         : {distance.shape}")

Shape         : (89198, 6)


In [5]:
from sklearn.neighbors import BallTree

seuil_km = 1.0
seuil_rad = seuil_km / 6371.0 


all_rad = np.radians(data[["latitude", "longitude"]].values)
climat_rad = np.radians(distance[["latitude", "longitude"]].values)


tree = BallTree(climat_rad, metric="haversine")


dist, idx = tree.query(all_rad, k=1)


dist_km = dist[:, 0] * 6371.0

distance_cols = distance.columns.difference(["longitude", "latitude"])
merged = data.copy()
merged[distance_cols] = distance.iloc[idx[:, 0]][distance_cols].values

merged["dist_distance_km"] = dist_km
merged["match_distance "] = (dist[:, 0] <= seuil_rad).astype(int)

print("Total all_points:", len(data))
print("Match <= seuil:", merged["match_distance "].sum())
print("No match:", (merged["match_distance "] == 0).sum())
print("Distance min/max (km):", merged["dist_distance_km"].min(), merged["dist_distance_km"].max())


merged.loc[merged["match_distance "] == 0, distance_cols] = np.nan

merged.drop(columns=["dist_distance_km", "match_distance "], inplace=True)

Total all_points: 89198
Match <= seuil: 89198
No match: 0
Distance min/max (km): 0.0 0.0


In [7]:
distance.columns

Index(['latitude', 'longitude', 'landcover', 'dist_residentiel_m',
       'dist_route_m', 'dist_eau_m'],
      dtype='object')

In [6]:
merged.columns

Index(['latitude', 'longitude', 'landcover', 'COARSE', 'SAND', 'SILT', 'CLAY',
       'TEXTURE_USDA', 'TEXTURE_SOTER', 'BULK', 'ORG_CARBON', 'PH_WATER',
       'CN_RATIO', 'CEC_SOIL', 'CEC_CLAY', 'CEC_EFF', 'BSAT', 'ALUM_SAT',
       'ESP', 'TCARBON_EQ', 'GYPSUM', 'ELEC_COND', 'in_fire', 'prec_ete',
       'tmax_ete', 'tmax_hiver', 'tmin_ete', 'Elevation',
       'dif_perc_hiver_printemps', 'dif_perc_automne_printemps',
       'dif_perc_ete_automne', 'dif_max_automne_printemps',
       'dif_max_hiver_printemps', 'dif_min_hiver_printemps',
       'dif_min_automne_hiver', 'dif_max_hiver_min_automne', 'dist_eau_m',
       'dist_residentiel_m', 'dist_route_m'],
      dtype='object')

In [8]:
merged.to_csv("/content/drive/MyDrive/collab/f2/grid_portugal_+india_features.csv", index=False)